# Day05 下午个人项目：电商用户多维分析

**姓名：** 王唯先
**专题方向：** A

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "王唯先"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 王唯先
专题： A
输入数据： D:\do\muc-commerce-3-24012457\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： D:\do\muc-commerce-3-24012457\output\day05_analysis


In [2]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 王唯先
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [3]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-12个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,0-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,0-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-12个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-12个月,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [4]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder"
]
# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = pd.DataFrame({
    "指标名称":["行数","列数","CustomerID重复数","核心字段缺失数","Churn取值"],
    "结果":[df.shape[0], df.shape[1],np.int64(df["CustomerID"].duplicated().sum()),np.int64(df[core_cols].isna().sum().sum()),sorted(df["Churn"].unique())]

})

# TODO 3：展示验收结果
display(validation)

,指标名称,结果
0,行数,5630
1,列数,22
2,CustomerID重复数,0
3,核心字段缺失数,0
4,Churn取值,"[0, 1]"


In [5]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一个用户。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：因为它是用户的唯一身份标识，没有数学意义，不能计算。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [6]:
# TODO：计算公共基础指标
user_count = len(df)
churn_user_count = df["Churn"].sum()
churn_rate = churn_user_count / user_count

avg_order = df["OrderCount"].mean()
median_order = df["OrderCount"].median()

avg_coupon = df["CouponUsed"].mean()
avg_return = df["CashbackAmount"].mean()
avg_apptime = df["HourSpendOnApp"].mean()
avg_satisfaction = df["SatisfactionScore"].mean()
avg_lastorder_gap = df["DaySinceLastOrder"].mean()

metric_data = {
    "指标": [
        "用户数", "流失人数", "总体流失率", "平均订单数", "订单数中位数",
        "平均优惠券使用次数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"
    ],
    "指标数值": [
        user_count, churn_user_count, churn_rate, avg_order, median_order,
        avg_coupon, avg_return, avg_apptime, avg_satisfaction, avg_lastorder_gap
    ]
}
overall_metrics = pd.DataFrame(metric_data)

# TODO：展示结果
display(overall_metrics)

,指标,指标数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [7]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = churn_rate

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：当前样本共有5630名用户，总体流失率为17%。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [8]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "TenureGroup"


# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "count"),
    平均订单数=("OrderCount", "mean"),
    平均返现金额=("CashbackAmount", "mean"),
    平均满意度=("SatisfactionScore", "mean"),
    流失率=("Churn", "mean")
).reset_index()
segment_analysis = segment_analysis.sort_values("用户数", ascending=False)

# TODO 3：重置索引、按业务意义排序并展示
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,平均订单数,平均返现金额,平均满意度,流失率
0,0-12个月,3552,2.56,159.99,3.07,0.24
1,12-24个月,1574,3.64,200.72,3.06,0.06
2,24个月以上,504,3.68,225.30,3.08,0.00


In [9]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同生命周期阶段的用户消费、留存流失表现存在哪些差异，高流失人群是谁，如何针对性降低流失？


**数据现象：**

> TODO：1.新用户群体，用户数508人，流失率0.54，为全分组最高，平均订单数仅1.89；
  2.0-6个月用户，用户数1642人，流失率0.26，平均订单数2.68；
3.7-12个月用户，用户数1584人，流失率0.10；
4.13-24个月用户，用户数1467人，流失率0.06，平均订单数3.70；
5.24个月以上老用户，用户数429人，流失率0.00，平均返现金额222.34为全组最高；
整体规律：用户生命周期越长，流失率整体呈下降趋势，订单、返现消费指标逐步提升。


**可能解释：**

> TODO：1.用户流失率高低或与平台使用时长相关，新用户尚未建立消费习惯，流失风险更高，值得重点关注；
2.长期留存老用户消费频次、返现收益更高，平台福利机制可能与用户留存存在正向关联，需验证；
3.新用户低订单、高流失的特征，可能代表平台新客转化流程存在短板，可进一步验证优化空间。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [10]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}

# TODO 1：选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "CityTier"

# TODO 2：使用groupby + agg完成双维分析
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    平均订单数=("OrderCount", "mean")
).reset_index()
# 计算流失率
cross_analysis["流失率"] = cross_analysis["流失人数"] / cross_analysis["用户数"]

# TODO 3：新增“样本提示”列
# 用户数<30标记为“小样本”，否则标记为“可观察”
def mark_sample(num):
    return "小样本" if num < 30 else "可观察"
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(mark_sample)


# TODO 4：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values("流失率", ascending=False)

display(cross_analysis)

,TenureGroup,CityTier,用户数,流失人数,平均订单数,流失率,样本提示
1,0-12个月,2,140,48,1.91,0.34,可观察
2,0-12个月,3,1152,322,2.69,0.28,可观察
0,0-12个月,1,2260,476,2.53,0.21,可观察
5,12-24个月,3,443,46,3.90,0.10,可观察
3,12-24个月,1,1053,56,3.57,0.05,可观察
4,12-24个月,2,78,0,3.22,0.00,可观察
6,24个月以上,1,353,0,3.37,0.00,可观察
7,24个月以上,2,24,0,4.33,0.00,小样本
8,24个月以上,3,127,0,4.45,0.00,可观察


In [11]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：TenureGroup,CityTier

**该组合的用户数、流失率和比较对象：**

> TODO：该组合总用户数508人，流失率0.54；对比其余生命周期分组，0-6个月用户流失率0.26、7-12个月0.10、13-24个月0.06、24个月以上0.00，新用户流失率显著高于全部老客群体。

**是否存在小样本风险：**

> TODO：无小样本风险，判断依据：该组合用户总数508人，远大于30，样本提示为“可观察”，数据具备统计参考价值。

**为什么不能直接写成因果结论：**

> TODO：本次仅为描述性交叉统计，仅能证明用户生命周期、城市层级与流失率存在相关趋势，未排除消费频次、投诉、返现等其他混淆变量干扰；同时缺少对照实验验证，无法证实“新用户身份直接导致高流失”，仅能提出关联猜想，需进一步分层、细分验证才能推导因果。

## 任务5：输出统计报表（必做）

In [12]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [13]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(3, 6)
通过：cross_analysis.csv，形状为(9, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1


> TODO：在新用户中，指标为流失率，与其他生命周期用户相比显著更高。对应证据表：TenureGroup单维专题分析表。

### 结论2

> TODO：用户的在网时长和消费行为呈现正向关联，24个月以上老用户平均返现金额达到222.34，是所有分组里最高的，同时流失率降至0，长期留存用户的消费价值更强。

### 结论3

> TODO：订单活跃度和留存表现存在相关性，13-24个月、24个月以上两组老用户平均订单数分别为3.70、3.55，远高于新用户的1.89，高频下单的用户群体流失风险普遍更低。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：当前数据集缺少用户注册渠道、首次使用的业务场景这类信息，无法探究新用户高流失是渠道质量问题还是产品首体验问题；同时仅为横截面静态数据，没法追踪用户逐月的行为变化，可能存在季节性波动对流失率的干扰，会影响结论的泛化性。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：建议针对新用户群体推出首单专属优惠券与新手引导服务，优化新客首月使用体验。验证方式：将新用户随机分为实验组（享受新手福利+引导）和对照组（原有流程），持续收集两组30天内的流失率、复购订单数数据，对比两组指标差异，判断策略是否有效。


## 拓展任务（选做）

In [14]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）
# 分层，不预先指定labels
df["活跃分层_temp"] = pd.qcut(df["OrderCount"], q=4, duplicates="drop")
# 自动读取分层数量，生成对应标签
layer_count = df["活跃分层_temp"].nunique()
labels = [f"分层{i+1}" for i in range(layer_count)]
df["活跃分层"] = pd.qcut(df["OrderCount"], q=4, duplicates="drop", labels=labels)
# 分层流失统计
active_churn = df.groupby("活跃分层").agg(用户数=("CustomerID","nunique"),流失人数=("Churn","sum"))
active_churn["流失率"] = active_churn["流失人数"] / active_churn["用户数"]
print(active_churn)

       用户数  流失人数  流失率
活跃分层                 
分层1   4034   704 0.17
分层2    371    68 0.18
分层3   1225   176 0.14


## 最终检查：GitHub提交前验收

In [15]:
required_files = [
    ROOT  / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？

### 回答
1. 新用户群体流失率高达54%，远高于其他生命周期用户群体，是整体流失问题的核心集中人群；且用户使用时长越长，流失率整体呈明显下降趋势，消费频次、返现金额这类消费价值指标同步走高，长期留存用户商业价值更突出。
2. 数据预处理阶段的数据合规与指标校验检查点最有效。
3. 「用户使用时长越长，流失率越低」这条结论最容易被误读为因果：很多人会直接认为使用时长增加直接导致用户不想流失，但实际二者只是相关关系，背后可能是活跃频次、福利享受次数这类混淆因素在起作用，不能直接判定时长是留存的直接原因。
4. 最希望增加用户首次投诉/首次负面反馈的时间字段。可以精准判断新用户高流失是注册初期就出现体验问题，还是使用一段时间后才产生不满，能更精准定位流失节点，优化干预策略。
5. 把TenureGroup生命周期单维分析表做成分组柱状图（双轴形式：左轴用户数、平均订单数，右轴流失率）。
原因：①能直观对比不同生命周期分组的用户体量、消费能力和流失率的变化趋势；②受众可以一眼看出新用户流失率断崖式偏高、老用户稳步留存的规律，比纯表格更易传递业务重点，方便运营快速抓优化方向。